# B1.1 · The loop, and the verifier that decides what it may conclude

**Function B — Application Security with an AI SDLC → The Agentic Harness**  ·  *Both directions*

Builds on **[B1.0 · What an agentic harness is](https://spbreed.github.io/cyber-commons/lessons/B1.0.html)**.

| | |
|---|---|
| Tools used | pytest |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The same model, the same three proposals, the same order. Swap only the verifier and one run fixes the refund check while the other ships an off-by-one that refunds a traveller twice — with a clean trace, reporting success. A weak verifier does not fail loudly; it succeeds incorrectly.

> **At CyberTravels.** The Coding Agent is asked to fix the refund-eligibility check. Whether the fix ships depends entirely on what the loop is allowed to believe about it, and "the model said it was done" is not a check. R7.

## 2 · The framework

```
   plan  ---> act  ---> verify ---> stop
     ^                     |
     +---------------------+  (not verified: go again, within budget)

   swap ONLY the verifier and the same proposals give:

     behavioural test   ->  attempt 2 accepted, refund check correct
     "the model says"   ->  attempt 1 accepted, off-by-one shipped

   fooled by:            available:
     behavioural test    changing real behaviour     when executable
     exact-match oracle  nothing (needs the answer)  rarely
     shape check         any well-formed output      ALWAYS
     LLM judge           confident prose             ALWAYS

   the two available everywhere are the two weakest,
   and they do not error -- they APPROVE
```

A harness is four moves in a loop:

1. **Plan** — the model proposes what to do next.
2. **Act** — the harness executes that proposal against a tool.
3. **Verify** — something decides whether the result is acceptable.
4. **Stop** — either verification succeeded, or a budget ran out.

That is the whole architecture. Everything that makes a harness safe or unsafe
lives in moves 3 and 4, and this chapter spends most of its time there.

The reason to build it explicitly rather than adopt a framework is that
frameworks make moves 1 and 2 easy and leave 3 and 4 as your problem — usually
with a default of "the model says it's done" and "loop forever". You need to
know exactly what yours does.

### The verifier decides what the harness is allowed to believe

State it plainly, because the rest of Function B depends on it. A harness with
a weak verifier does not fail loudly. It **succeeds incorrectly**, produces a
clean trace, and the failure is discovered downstream — in CyberTravels' case,
by a traveller who was refunded twice.

Verifiers form a hierarchy, ordered by what it takes to fool them:

| Verifier | Fooled by | Available when |
|---|---|---|
| **Behavioural test** | changing real behaviour | you can execute the thing |
| **Exact-match oracle** | nothing, but needs the answer up front | rarely |
| **Shape check** | any well-formed output | always |
| **LLM judge** | confident prose | always |

The trap is that the two available-everywhere options are the two weakest, and
they fail in the worst possible direction: they do not error, they **approve**.

There is also a subtler failure worth seeing rather than reading about: a
verifier that is *correct* but reads stale state. A test runner importing cached
bytecode reports on code that is no longer on disk. A lying oracle is worse than
no oracle, because you stop looking.

The demo below fixes a refund-eligibility check for the Workflow Agent, and then
changes exactly one thing — the verifier — so the same model, the same
proposals and the same order produce opposite outcomes.

## 3 · The model backend, and the plan it produces

In [ ]:
# --- model backend: replay by default, real model when you configure one ----
# Nothing here is Anthropic- or vendor-specific beyond one URL and one header
# shape. Standard library only, so the notebook stays self-contained.
import json, os, urllib.error, urllib.request

# The cheapest current model on each side, which is what a lesson needs.
FRONTIER_DEFAULT   = "claude-haiku-4-5-20251001"
OPEN_WEIGHT_DEFAULT = "glm-4.6"
TIMEOUT = 60

def _kaggle_secret(name):
    """On Kaggle, a key lives in Add-ons -> Secrets rather than the environment.

    kaggle_secrets is pre-installed in the Kaggle image and absent everywhere
    else, so the import is guarded and the notebook needs no dependency. It also
    requires the notebook to have internet enabled, which on Kaggle requires a
    phone-verified account - see the note printed below.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("ANTHROPIC_API_KEY") or _kaggle_secret("ANTHROPIC_API_KEY"):
        os.environ.setdefault("ANTHROPIC_API_KEY",
                              os.environ.get("ANTHROPIC_API_KEY")
                              or _kaggle_secret("ANTHROPIC_API_KEY") or "")
        return "frontier", os.environ.get("MODEL", FRONTIER_DEFAULT)
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _anthropic(prompt, system, model, max_tokens, temperature):
    body = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]}
    if system:
        body["system"] = system
    headers = {"x-api-key": os.environ["ANTHROPIC_API_KEY"],
               "anthropic-version": "2023-06-01"}
    # An identity-linked key is scoped to a workspace and the API refuses the
    # call without being told which one. A plain organisation key needs nothing
    # here, so the header is only sent when it is set.
    ws = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    if ws:
        headers["anthropic-workspace-id"] = ws
    base = os.environ.get("ANTHROPIC_BASE_URL", "https://api.anthropic.com").rstrip("/")
    out = _post(f"{base}/v1/messages", body, headers)
    return "".join(b.get("text", "") for b in out.get("content", [])).strip()

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        fn = _anthropic if kind == "frontier" else _openai_compatible
        return fn(prompt, system, model, max_tokens, temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        # Print what the API actually said. "failed: 400" costs whoever hits
        # this an hour; the body usually names the exact missing header or
        # parameter, and it never contains the key.
        detail = getattr(e, "code", None) or type(e).__name__
        why = ""
        if hasattr(e, "read"):
            try:
                why = json.loads(e.read().decode()).get("error", {}).get("message", "")
            except Exception:
                why = ""
        print(f"   !! {kind} backend ({model}) failed: {detail}"
              f"{' - ' + why if why else ''}")
        print("      Using the replay, which is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, set one of:")
    print()
    print("   frontier     export ANTHROPIC_API_KEY=...   # cheapest: " + FRONTIER_DEFAULT)
    print("                (an identity-linked key also needs")
    print("                 ANTHROPIC_WORKSPACE_ID=...)")
    print("   open weight  export OPENAI_BASE_URL=http://localhost:11434/v1 \\")
    print("                       OPENAI_API_KEY=ollama MODEL=glm-4.6")
    print()
    print("   On Kaggle: Add-ons -> Secrets, add ANTHROPIC_API_KEY, and switch")
    print("   Internet on in the notebook settings. Internet requires a")
    print("   phone-verified Kaggle account; without it DNS fails in the kernel")
    print("   and this lesson correctly stays on the replay.")

## 4 · The same lesson, against a real model

Everything below this point runs identically on three backends. Offline it uses
a deterministic replay that is labelled as a replay wherever it appears — never
presented as a model's output. With `ANTHROPIC_API_KEY` set it calls a frontier
model; with `OPENAI_BASE_URL` set it calls any OpenAI-compatible endpoint,
which covers Ollama, vLLM and the hosted open-weight providers.

The point of running it both ways is not that the answers match. It is that
**the lesson's assertion holds either way** — if it only holds against the
replay, the lesson was testing the replay.

In [ ]:
TASK = 'Plan how to determine whether a reported SQL injection in orders/report.py is reachable from an unauthenticated HTTP request. Give at most four numbered steps, each one an action, not a thought.'

REPLAY = "1. Locate the HTTP route that reaches orders/report.py.\n2. Check whether that route requires authentication.\n3. Trace request.args['ref'] from the handler to db.execute.\n4. Send one request with a benign marker and observe the query log."

answer, used, model = ask(TASK, replay=REPLAY,
            system='You produce plans for a security harness. Numbered actions only.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("produced numbered steps", any(answer.strip().startswith(p) for p in ("1.", "1)", "- 1")))
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, three possible backends. Offline the answer")
print("is the replay and is labelled as one; with a key it is the model's.")

## 5 · Demo — the loop, with a task that has a right answer

The task: fix a function that mis-computes a security-relevant value. The model produces three attempts, the last of which is correct.

In [ ]:
import time
from dataclasses import dataclass, field

class ReplayModel:
    """DETERMINISTIC REPLAY — not a language model.
    Emits a fixed sequence so the loop's control flow is the thing under test."""
    def __init__(self, proposals, name="replay"):
        self.proposals, self.name, self.calls = list(proposals), name, 0
    def propose(self, _prompt):
        # after the script runs out it repeats — exactly what a stuck loop does
        p = self.proposals[min(self.calls, len(self.proposals) - 1)]
        self.calls += 1
        return p

@dataclass
class Step:
    n: int; proposal: str; ok: bool; detail: str; ms: float = 0.0

@dataclass
class Trace:
    steps: list = field(default_factory=list)
    stopped_by: str = ""
    succeeded: bool = False
    def table(self):
        rows = [f"{'step':>4}  {'ok':<6}{'proposal':<44}detail",
                f"{'-'*4}  {'-'*6}{'-'*44}{'-'*28}"]
        for s in self.steps:
            rows.append(f"{s.n:>4}  {str(s.ok):<6}{s.proposal[:44]:<44}{s.detail[:28]}")
        rows.append(f"\nstopped by: {self.stopped_by}    succeeded: {self.succeeded}")
        return "\n".join(rows)

def run(model, verifier, goal="", max_steps=5, max_seconds=10.0):
    tr, started = Trace(), time.monotonic()
    for n in range(1, max_steps + 1):
        t0 = time.monotonic()
        proposal = model.propose(f"{goal} (attempt {n})")      # PLAN
        ok, detail = verifier(proposal)                        # ACT + VERIFY
        tr.steps.append(Step(n, proposal, ok, detail, (time.monotonic()-t0)*1000))
        if ok:                                                 # STOP
            tr.stopped_by, tr.succeeded = "verifier satisfied", True
            return tr
        if time.monotonic() - started > max_seconds:
            tr.stopped_by = f"time budget ({max_seconds}s)"
            return tr
    tr.stopped_by = f"step budget ({max_steps} steps)"
    return tr

ATTEMPTS = [
 "def is_expired(cert): return cert.days_left < 0",     # off-by-one: 0 is expired
 "def is_expired(cert): return cert.days_left <= 0",    # correct
]

In [ ]:
# The verifier: execute the proposal against known-good cases.
CASES = [(-5, True), (0, True), (1, False), (30, False)]

class Cert:
    def __init__(self, d): self.days_left = d

def behavioural_verifier(src):
    ns = {}
    try:
        exec(compile(src, "<proposal>", "exec"), ns)
        fn = ns["is_expired"]
    except Exception as e:
        return False, f"did not compile: {type(e).__name__}"
    for days, expected in CASES:
        got = fn(Cert(days))
        if got != expected:
            return False, f"is_expired(days_left={days}) → {got}, want {expected}"
    return True, f"all {len(CASES)} cases pass"

tr = run(ReplayModel(ATTEMPTS), behavioural_verifier,
         goal="fix certificate expiry check", max_steps=5)
print(tr.table())

## 6 · Where it breaks — change only the verifier

Same model. Same proposals. Same order. The only difference is what the loop believes when it decides it has succeeded.

In [ ]:
def self_grading_verifier(src):
    """The model judges its own work. Ships in a lot of harnesses."""
    looks_done = bool(src.strip()) and src.strip().startswith("def ")
    return looks_done, "judge: looks like a valid fix, approving"

tr2 = run(ReplayModel(ATTEMPTS), self_grading_verifier,
          goal="fix certificate expiry check", max_steps=5)
print(tr2.table())

ns = {}; exec(compile(tr2.steps[-1].proposal, "<x>", "exec"), ns)
print(f"\nthe accepted code says a cert with 0 days left is expired: "
      f"{ns['is_expired'](Cert(0))}")
print("It is not. A certificate expiring today is still valid today, and this")
print("harness just shipped that. The trace above is clean and reports success.")

## 7 · The control — the verifier is a security control

State it plainly, because the rest of the track depends on it: **the verifier decides what the harness is allowed to believe.** A harness with a weak verifier does not fail loudly. It succeeds incorrectly, produces a clean trace, and the failure is discovered downstream.

In [ ]:
def compare(model_factory, verifiers, **kw):
    out = {}
    for name, v in verifiers.items():
        tr = run(model_factory(), v, **kw)
        out[name] = {"succeeded": tr.succeeded, "steps": len(tr.steps),
                     "stopped_by": tr.stopped_by,
                     "accepted": tr.steps[-1].proposal if tr.succeeded else None}
    return out

def no_verifier(_src):
    return False, "no verifier configured"

results = compare(lambda: ReplayModel(ATTEMPTS),
                  {"behavioural (executes the code)": behavioural_verifier,
                   "self-grading (asks the model)":   self_grading_verifier,
                   "none":                            no_verifier},
                  goal="fix expiry check", max_steps=4)
print(f"{'verifier':34s}{'succeeded':11s}{'steps':7s}stopped by")
print("-" * 76)
for name, r in results.items():
    print(f"{name:34s}{str(r['succeeded']):11s}{r['steps']:<7}{r['stopped_by']}")

print("\nwhat each one accepted:")
for name, r in results.items():
    print(f"   {name:34s}{r['accepted'] or '—'}")
assert results["behavioural (executes the code)"]["accepted"] == ATTEMPTS[1]
assert results["self-grading (asks the model)"]["accepted"] == ATTEMPTS[0]

## 8 · Four verifiers, ranked by what it takes to fool them

The loop above used the strongest kind available. Most harnesses cannot, so it is worth seeing all four against the same input, and seeing which two are available everywhere.

In [ ]:
BROKEN  = "def parse_port(s): return int(s)"          # accepts 0, 99999, -1
CORRECT = ("def parse_port(s):\n"
           "    p = int(s)\n"
           "    if not (1 <= p <= 65535): raise ValueError('port out of range')\n"
           "    return p")

def behavioural(src):
    ns = {}
    try:
        exec(compile(src, "<p>", "exec"), ns); fn = ns["parse_port"]
    except Exception as e:
        return False, f"compile failed: {e}"
    for bad in ("0", "70000", "-1"):
        try:
            fn(bad)
            return False, f"accepted out-of-range port {bad!r}"
        except ValueError:
            pass
    try:
        if fn("443") != 443:
            return False, "rejected a valid port"
    except Exception as e:
        return False, f"valid port raised {e}"
    return True, "rejects out-of-range, accepts valid"

def exact_match(expected):
    return lambda src: (src.strip() == expected.strip(),
                        "exact match" if src.strip() == expected.strip()
                        else "differs from the reference implementation")

def shape_check(src):
    ok = src.strip().startswith("def parse_port")
    return ok, "defines parse_port" if ok else "wrong shape"

def llm_judge(src):
    ok = bool(src.strip()) and not src.lower().startswith("i cannot")
    return ok, "judge: this looks like a correct implementation"

VERIFIERS = {"behavioural (executes it)": behavioural,
             "exact-match oracle":        exact_match(CORRECT),
             "shape check":               shape_check,
             "llm judge":                 llm_judge}

print(f"{'verifier':28s}{'on BROKEN':12s}{'on CORRECT':12s}detail (broken)")
print("-" * 84)
for name, v in VERIFIERS.items():
    b_ok, b_why = v(BROKEN)
    c_ok, _     = v(CORRECT)
    print(f"{name:28s}{str(b_ok):12s}{str(c_ok):12s}{b_why[:32]}")

## 9 · Where it breaks — the exact-match oracle is also wrong

Look at the `on CORRECT` column. The behavioural verifier is the only one that gets *both* right. The exact-match oracle rejects a correct implementation that differs from its reference — which is why nobody uses it, and why teams fall back to the two weak options.

In [ ]:
ALTERNATIVE = ("def parse_port(s):\n"
               "    p = int(s)\n"
               "    if p < 1 or p > 65535:\n"
               "        raise ValueError('bad port')\n"
               "    return p")
print("a correct implementation, written differently:")
for name, v in VERIFIERS.items():
    ok, why = v(ALTERNATIVE)
    print(f"   {name:28s}{str(ok):7s}{why[:44]}")
print("\nThe oracle says no. Behavioural says yes. Only one of those is useful")
print("on code you did not write in advance.")

## 10 · The subtler failure — a correct verifier reading stale state

This one is not about weak checks. The check is right; the *input* to it is stale. Python's bytecode cache reproduces it faithfully.

In [ ]:
import os, sys, tempfile, subprocess, textwrap, shutil, pathlib

work = pathlib.Path(tempfile.mkdtemp())
(work / "mod.py").write_text("def check(x):\n    return True   # broken: always passes\n")
(work / "test_mod.py").write_text(textwrap.dedent("""
    from mod import check
    def test_rejects_bad():
        assert check(-1) is False
"""))

def run_check(workdir, clear_cache):
    if clear_cache:
        shutil.rmtree(workdir / "__pycache__", ignore_errors=True)
    env = {**os.environ, "PYTHONPATH": str(workdir)}
    if clear_cache:
        env["PYTHONDONTWRITEBYTECODE"] = "1"
    r = subprocess.run([sys.executable, "-c",
                        "import mod; print('PASS' if mod.check(-1) is False else 'FAIL')"],
                       cwd=workdir, env=env, capture_output=True, text=True)
    return r.stdout.strip()

print("1. verifier on the broken code:      ", run_check(work, clear_cache=True))
# the agent "fixes" it
(work / "mod.py").write_text("def check(x):\n    return x >= 0\n")
print("2. after a real fix, cache cleared:  ", run_check(work, clear_cache=True))
# now put the broken version back, but leave a stale cache in place
import py_compile
(work / "mod.py").write_text("def check(x):\n    return True   # broken again\n")
py_compile.compile(str(work / "mod.py"), doraise=True)
(work / "mod.py").write_text("def check(x):\n    return x >= 0\n")
os.utime(work / "mod.py", (0, 0))          # make the source look older than the cache
print("3. source fixed, STALE cache honoured:", run_check(work, clear_cache=False),
      " ← the verifier is reading code that is not on disk")
shutil.rmtree(work, ignore_errors=True)

## 11 · The control — rank verifiers and always clear derived state

In [ ]:
RANKING = [
 (1, "behavioural / property test", "must change observable behaviour",
     "execute the artefact against facts that must hold"),
 (2, "differential test",           "must match a trusted second implementation",
     "run old and new against the same inputs"),
 (3, "exact-match oracle",          "nothing — but needs the answer in advance",
     "only usable on a fixed corpus"),
 (4, "shape / schema check",        "any well-formed output",
     "use for conformance ONLY, never for quality — see B1.4"),
 (5, "llm judge",                   "confident prose",
     "acceptable only as a filter before a real check, never as the last word"),
]
print(f"{'rank':5s}{'verifier':30s}{'fooled by':44s}")
print("-" * 80)
for r, name, fooled, use in RANKING:
    print(f"{r:<5}{name:30s}{fooled:44s}")
    print(f"{'':35s}{use}")

CHECKLIST = [
 "does it EXECUTE the artefact, or only inspect it?",
 "would it fail if the artefact were subtly wrong?",
 "does it clear caches / derived state before reading?",
 "is its own correctness tested (does it fail on known-bad input)?",
]
print("\nverifier review checklist:")
for c in CHECKLIST:
    print("   ·", c)

# the fourth item, applied to our own verifier
assert behavioural(BROKEN)[0] is False, "verifier must fail on known-bad"
assert behavioural(CORRECT)[0] is True, "verifier must pass on known-good"
assert behavioural(ALTERNATIVE)[0] is True, "verifier must accept alternatives"
print("\nour behavioural verifier passes its own test: fails broken, accepts both correct forms.")

## What you just proved

The loop fixes a refund-eligibility check in two attempts and stops when the behavioural verifier is satisfied. The identical model and proposals, with only the verifier swapped for one that asks the model whether it is done, accept the off-by-one on the first attempt and report success with a clean trace. Four verifiers are then run against one malformed output: the shape check and the judge both approve it, the exact-match oracle is itself wrong, and a correct verifier reading stale bytecode passes code that is no longer on disk.

## Your turn

Name your harness's verifier out loud. If the sentence contains "the model checks" or "it looks right", you have a judge, and a judge approves confident prose. Then check the second thing: does anything clear derived state between attempts?

---

**Next → [B1.2 · What the loop may touch: tools, depth and doing it twice](https://spbreed.github.io/cyber-commons/lessons/B1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*